# Ruh sağlığı tahmini

Amacımız, kişinin demografik, kariyer/akademik ve yaşam tarzı verilerine bakarak depresyon eğilimi olup olmadığını (0: Yok, 1: Var) tahmin etmektir.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, accuracy_score
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
train= pd.read_csv('train(6).csv')
test= pd.read_csv('test(6).csv')

In [3]:
train.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


In [4]:
test.head()

,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness
0,140700,Shivam,Male,53.0,Visakhapatnam,Working Professional,Judge,NaN,2.0,NaN,NaN,5.0,Less than 5 hours,Moderate,LLB,No,9.0,3.0,Yes
1,140701,Sanya,Female,58.0,Kolkata,Working Professional,Educational Consultant,NaN,2.0,NaN,NaN,4.0,Less than 5 hours,Moderate,B.Ed,No,6.0,4.0,No
2,140702,Yash,Male,53.0,Jaipur,Working Professional,Teacher,NaN,4.0,NaN,NaN,1.0,7-8 hours,Moderate,B.Arch,Yes,12.0,4.0,No
3,140703,Nalini,Female,23.0,Rajkot,Student,NaN,5.0,NaN,6.84,1.0,NaN,More than 8 hours,Moderate,BSc,Yes,10.0,4.0,No
4,140704,Shaurya,Male,47.0,Kalyan,Working Professional,Teacher,NaN,5.0,NaN,NaN,5.0,7-8 hours,Moderate,BCA,Yes,3.0,4.0,No


In [5]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140700 entries, 0 to 140699
Data columns (total 20 columns):
 #   Column                                 Non-Null Count   Dtype  
---  ------                                 --------------   -----  
 0   id                                     140700 non-null  int64  
 1   Name                                   140700 non-null  object 
 2   Gender                                 140700 non-null  object 
 3   Age                                    140700 non-null  float64
 4   City                                   140700 non-null  object 
 5   Working Professional or Student        140700 non-null  object 
 6   Profession                             104070 non-null  object 
 7   Academic Pressure                      27897 non-null   float64
 8   Work Pressure                          112782 non-null  float64
 9   CGPA                                   27898 non-null   float64
 10  Study Satisfaction                     27897 non-null   

In [6]:
HEDEF = 'Depression'

In [7]:
# Gereksiz Sütunları Düşürme (İsim tahminde kullanılamaz)
if 'Name' in train.columns: train.drop(columns=['Name'], inplace=True)
if 'Name' in test.columns: test.drop(columns=['Name'], inplace=True)


In [8]:
# 3. Kategorik (Metinsel) Sütunları Güvenle Sayısallaştırma
# Hedef kolonu muaf tutarak sadece girdi metin sütunlarını topluyoruz
#Çok sayıdaki kategorik/metinsel sütunu (Gender, City, Profession, Degree, Have you ever had suicidal thoughts ? vb.)
#OrdinalEncoder ile güvenli şekilde sayısallaştırmak.

kategorik_sutunlar = [
    kolon for kolon in train.select_dtypes(include=['object']).columns 
    if kolon != HEDEF
]

In [9]:
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
train[kategorik_sutunlar] = encoder.fit_transform(train[kategorik_sutunlar].astype(str))
test[kategorik_sutunlar] = encoder.transform(test[kategorik_sutunlar].astype(str))


In [10]:
# Psikometrik Özellik Mühendisliği (Stres ve Yaşam Standartları)
for df in [train, test]:
    # Basınç ve Memnuniyet Etkileşimi (NaN veya Boş değer korumalı)
    df['total_pressure'] = df['Academic Pressure'].fillna(0) + df['Work Pressure'].fillna(0)
    df['total_satisfaction'] = df['Study Satisfaction'].fillna(0) + df['Job Satisfaction'].fillna(0)
    
    # Zaman Baskısı Endeksi: Çalışma saati arttıkça ve uyku azaldıkça psikolojik yük artar
    # Metinsel gelebilecek uyku sürelerini sayıya çevirmek adına dolaylı koruma
    df['sleep_num'] = pd.to_numeric(df['Sleep Duration'], errors='coerce').fillna(7)
    df['work_hours_num'] = pd.to_numeric(df['Work/Study Hours'], errors='coerce').fillna(8)
    df['time_stress_ratio'] = df['work_hours_num'] / (df['sleep_num'] + 1e-5)
    
    # Finansal ve Ailevi Risk Skoru
    df['family_financial_risk'] = df['Financial Stress'].fillna(0) * (df['Family History of Mental Illness'].fillna(0) + 1)

print("2. Adım: Özellik mühendisliği tamamlandı. Modelleme başlatılıyor...")


2. Adım: Özellik mühendisliği tamamlandı. Modelleme başlatılıyor...


In [11]:
# Girdileri (X) ve Hedefi (y) Kesin Olarak Ayırma
giris_sutunlari = [kolon for kolon in train.columns if kolon not in ['id', HEDEF, 'Sleep Duration', 'Work/Study Hours']]

X = train[giris_sutunlari]
y = train[HEDEF]
X_test = test[giris_sutunlari] # Birebir aynı sütun dizilimi sağlandı

# 5-Fold Stratified Cross Validation Düzeni
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_tahminleri = np.zeros(len(train))
test_tahminleri = np.zeros(len(test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Psikolojik veri setlerinde aşırı öğrenmeyi (overfitting) engellemek için max_depth sınırlı tutulmuştur
    model = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.04,
        max_depth=6,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    
    # Olasılıkları kaydetme
    oof_tahminleri[val_idx] = model.predict_proba(X_val)[:, 1]
    test_tahminleri += model.predict_proba(X_test)[:, 1] / skf.n_splits
    
    fold_auc = roc_auc_score(y_val, oof_tahminleri[val_idx])
    print(f"Katman {fold + 1} ROC-AUC Skoru: {fold_auc:.4f}")

# Genel Başarı Oranı
genel_auc = roc_auc_score(y, oof_tahminleri)
print(f"\n---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: {genel_auc:.4f} <---")


Katman 1 ROC-AUC Skoru: 0.9749
Katman 2 ROC-AUC Skoru: 0.9735
Katman 3 ROC-AUC Skoru: 0.9749
Katman 4 ROC-AUC Skoru: 0.9767
Katman 5 ROC-AUC Skoru: 0.9747

---> TÜM VERİ SETİ GENEL ROC-AUC SKORU: 0.9750 <---


In [12]:
# Oof tahminlerini sınıfa çevirerek doğruluk oranı
oof_siniflar = (oof_tahminleri >= 0.5).astype(int)
print(f"---> GENEL DOĞRULUK (ACCURACY) ORANI: %{accuracy_score(y, oof_siniflar)*100:.2f} <---")


---> GENEL DOĞRULUK (ACCURACY) ORANI: %93.90 <---


In [13]:
# Kaggle Gönderi Dosyasının Oluşturulması
submission = pd.DataFrame({
    'id': test['id'],
    'Depression': (test_tahminleri >= 0.5).astype(int) # Yarışma tam sayı (0 veya 1) istiyorsa
})

In [14]:
submission.to_csv('submission_mental.csv', index=False)